# ARC-AGI-2 reference implementation &mdash; offline Kaggle kernel

**This notebook is a declared REFERENCE IMPLEMENTATION. It is not our work.**

- Upstream: `mikelou1/arc-agi2-lb33-89-minimal-perfpatch` &mdash; the field-leading public ARC-AGI-2
  pipeline (the canonical notebook currently returns 403 on pull).
- Fork used: `manderson240/arc-agi-2-fork-lb33-89-20260903` &mdash; public, measured **31.81** (rank 211).
- Fork provenance: its own first cell states it is *"Forked verbatim from ... by mikelou1"* and that
  *"No hyperparameters, code, or logic below this cell were changed from the source notebook"*. Its only
  patches are GPU-side logits normalisation and a target-token gather in `turbo_dfs` / `calc_scores`.

Everything below is upstream code. The paper must cite it as a reference implementation and must never
present it as a contribution of this project.

## Why this exists

The project held `0.00` on ARC-AGI-2. This kernel gives the ablation a measured comparison point.
The honest framing is a measured map, three points on one axis: a minimal public solver (0.84%), our
primitive ablation (4.18%), and this reference (the field's ~30 band). It buys Accuracy, not distinction:
745 of 2,172 teams already sit in the 30&ndash;40 band.

## What was changed to run offline (`enable_internet: False`)

1. **Base model by declaration, not download.** `model_sources` declares
   `sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1` (public Kaggle Models instance, Apache 2.0,
   7.27 GB, 42 votes). Upstream already loads it with `local_files_only=True`. Note that the base model is
   **not** raw `qwen-lm/qwen-3`: the grids15 SFT is part of the recipe and swapping in the un-tuned model
   would not be this pipeline.
2. **Dependencies by declaration, not download.** The image does not ship `unsloth`, `unsloth_zoo`, `trl`,
   `bitsandbytes`, `xformers`, `torchao>=0.13`, `cut_cross_entropy`, `tyro`, `msgspec` or `structlog`, and it
   ships `transformers` 5.0.0, `datasets` 5.0.0, `huggingface_hub` 1.11.0 and `dill` 0.4.1 at versions
   unsloth's dependency graph rejects. `dataset_sources` therefore declares
   `ser8147/arc2-unsloth-wheelhouse` &mdash; 14 pinned, hash-recorded wheels built by
   `scripts/kaggle/unsloth_wheelhouse.sh` &mdash; and the install cell runs
   `pip install --no-index --find-links` against it. The downgrades are forced, not cosmetic:
   `transformers` **5.0.0 &rarr; 4.57.6** (unsloth allows `>=4.51.3,<=5.5.0` but excludes 5.0.0/5.1.0; trl
   requires `>=4.56.1`, so 4.57.6 is the newest 4.x that satisfies both), `datasets` **5.0.0 &rarr; 4.3.0**
   (unsloth requires `<4.4.0`), `huggingface_hub` **1.11.0 &rarr; 0.36.2** (transformers 4.57.6 requires
   `<1.0`) and `dill` **0.4.1 &rarr; 0.4.0** (datasets 4.3.0 requires `<0.4.1`). Everything else the pipeline
   needs is already in the image at a satisfying version and is deliberately not pinned &mdash; an
   unnecessary pin only trips unrelated packages, e.g. `gcsfs` wants exactly `fsspec` 2025.3.0 and
   `opentelemetry-api` wants `importlib-metadata` <8.8.0. `torch` is deliberately **not** vendored: the
   image's own 2.10.0+cu128 satisfies unsloth's `torch<2.13.0,>=2.4.0`, and republishing it would clobber
   the image's CUDA build. The preflight prints the package table before the install and the install cell
   prints the same table afterwards, so the unblock is visible in one log.
3. **Accelerator is pinned to L4.** The training arguments are `bf16=True, fp16=False`, and among Kaggle's
   accelerators only L4 (Ada, SM 8.9) supports bf16. T4 and P100 cannot run this pipeline.
4. **Commit-mode selection.** `ARC_REFERENCE_TEST_SET=1` makes the non-rerun path read
   `arc-agi_test_challenges.json` and queue every test task, so the commit-mode artifact exercises the
   real competition code path and the real submission schema. It changes *which tasks run*, never how a
   task is solved. `global_end_time` is bounded in commit mode so verification stays inside quota; a
   competition rerun keeps the upstream `12h - 10min` budget unchanged.
5. **Failures are made loud.** Upstream calls `!python starter.py`, and a shell-magic failure does not stop
   the notebook &mdash; a broken run would silently continue and emit a placeholder submission. This kernel
   streams the worker log through a checked `subprocess` call instead. The same discipline covers the
   install: a missing wheelhouse, a wheelhouse that disagrees with the pinned set, a non-zero `pip` exit, a
   pin that did not take effect, or a failed import of `unsloth`/`trl`/`bitsandbytes` all raise before the
   solver starts.
6. **The submission is validated.** It is written to `/kaggle/working/submission.json` and checked against
   `arc-agi_test_challenges.json`: every task id present, both attempts present and well formed.

## The one declared deviation from the reference

**`use_gradient_checkpointing` is `True`, not upstream's `False`** &mdash; set at both declaration sites,
the `FastLanguageModel.from_pretrained` call and the `peft_params` dict passed to
`FastLanguageModel.get_peft_model`. It is the single upstream logic change made for memory, and it is a
pure memory-for-compute trade: the model sees the same data, the same sequence length and the same
augmentations. It also **ends this notebook's claim to be an unmodified reference**, so every number this
kernel produces must be reported with the deviation stated. The separate `gradient_checkpointing=False`
in `train_args` is a different `TrainingArguments` flag and is left at its upstream value.

**Not changed:** LoRA hyperparameters, DFS, `kgmon` ordering, augmentation counts (`n=16` train,
`n=2` eval), decode thresholds, the selection algorithm, the model path, or the competition input paths.
Gradient checkpointing is the single declared exception (see above), alongside the offline and commit-mode
adaptations in items 1-6.


Forked verbatim from https://www.kaggle.com/code/mikelou1/arc-agi2-lb33-89-minimal-perfpatch by mikelou1, for the Cohezion ARC-AGI-2 entry (manderson240).

License: no SPDX identifier is declared by the author in this notebook's metadata or cells. This fork relies instead on ARC Prize 2026 Competition Rules §2 ("Public Code Sharing") / §6b ("Use of Open Source"): code publicly shared on Kaggle associated with the competition (`competition_sources: arc-prize-2026-arc-agi-2`) is deemed licensed under an OSI-approved license by virtue of that sharing. No hyperparameters, code, or logic below this cell were changed from the source notebook.

# v38 — 基于原始 33.89 baseline 的最小性能补丁

本 notebook 以 `baseline_LB33.89_failed-in-aimo.ipynb` 为母版：**保留原有 9 个 cell、原 metadata、模型路径、竞赛输入、4×L4 并行、128 条 task-time 训练增强、16 条推理增强、LoRA 参数、解码阈值、候选聚合和 submission schema。**

仅在原 `arc_solver.py` 的两个 logits 热点中做定点替换：

1. `turbo_dfs`：仍对同一组 12 个 ARC token 使用同样的 NLL、阈值、候选排序与 DFS；仅把 full-vocabulary logits 的归一化留在 GPU，并只把 12 个 token 的结果传回 CPU。
2. `calc_scores`：仍使用同一 teacher-forced NLL 作为重排序分数；仅在 GPU 上 gather 目标 token 并关闭未被消费的 KV cache。

没有缩短或延长原 `global_end_time = now + 12h − 10min`，也没有改动 `starter.py`、结果写入、队列策略或提交逻辑。该版本的目的仅是减少 CPU↔GPU logits 传输与多余 KV cache 写入，从而在**不增加时间预算**的前提下提高已完成任务覆盖率。

In [ ]:
# ---------------------------------------------------------------------------
# Offline preflight -- the "before" state.
#
# Answers "can this pipeline satisfy its own dependencies with
# enable_internet: False?" from the log, BEFORE anything is installed. The
# wheelhouse cell below prints this exact table again once the install is done,
# so a single run shows the unblock line by line. This cell only measures; the
# hard gate lives after the install, where it can actually act on the result.
# ---------------------------------------------------------------------------
import glob
import importlib.metadata as md
import importlib.util
import os
import platform
import shutil
import subprocess
import sys
import time

PREFLIGHT_T0 = time.time()

print("=" * 78)
print("OFFLINE PREFLIGHT (before install)")
print("=" * 78)
print("python :", sys.version.split()[0], "|", platform.platform())
print("cwd    :", os.getcwd())
print("KAGGLE_IS_COMPETITION_RERUN :", os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
print("ARC_REFERENCE_TEST_SET      :", os.getenv("ARC_REFERENCE_TEST_SET"))


def dist_version(name):
    try:
        return md.version(name)
    except Exception as exc:
        return "NOT INSTALLED (%s)" % type(exc).__name__


# (display, import name, distribution name). The install cell prints this same
# table afterwards, so before and after can be compared line by line.
PACKAGES = [
    ("torch", "torch", "torch"),
    ("transformers", "transformers", "transformers"),
    ("unsloth", "unsloth", "unsloth"),
    ("unsloth_zoo", "unsloth_zoo", "unsloth_zoo"),
    ("peft", "peft", "peft"),
    ("trl", "trl", "trl"),
    ("bitsandbytes", "bitsandbytes", "bitsandbytes"),
    ("xformers", "xformers", "xformers"),
    ("datasets", "datasets", "datasets"),
    ("accelerate", "accelerate", "accelerate"),
    ("tokenizers", "tokenizers", "tokenizers"),
    ("huggingface_hub", "huggingface_hub", "huggingface_hub"),
    ("safetensors", "safetensors", "safetensors"),
    ("sentencepiece", "sentencepiece", "sentencepiece"),
    ("numpy", "numpy", "numpy"),
    ("tqdm", "tqdm", "tqdm"),
    ("flash_attn", "flash_attn", "flash-attn"),
    ("tensorflow", "tensorflow", "tensorflow"),
]


def package_table():
    """Print the table; return the display names that are not importable."""
    print("%-14s %-12s %s" % ("package", "importable", "version"))
    not_importable = []
    for display, module, distribution in PACKAGES:
        try:
            importable = importlib.util.find_spec(module) is not None
        except Exception as exc:                 # a broken/partial install
            importable = False
            print("%-14s   find_spec raised -> %s: %s" % ("", type(exc).__name__, exc))
        print("%-14s %-12s %s" % (display, importable, dist_version(distribution)))
        if not importable:
            not_importable.append(display)
    return not_importable


print("\n-- importable packages (find_spec, no import, no install) --")
before_missing = package_table()

print("\n-- full installed inventory (pip list --format=freeze) --")
listing = subprocess.run(
    [sys.executable, "-m", "pip", "list", "--format=freeze", "--disable-pip-version-check"],
    capture_output=True,
    text=True,
)
print(listing.stdout.strip() or listing.stderr.strip())

print("\n-- accelerators --")
try:
    import torch

    print("torch.cuda.is_available() :", torch.cuda.is_available())
    print("torch.cuda.device_count()  :", torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print("  gpu%d: %s sm_%d%d %.1f GiB bf16_supported=%s"
              % (index, props.name, props.major, props.minor,
                 props.total_memory / 1024 ** 3,
                 (props.major, props.minor) >= (8, 0)))
except Exception as exc:
    print("torch probe failed:", type(exc).__name__, exc)
print("nvidia-smi :", shutil.which("nvidia-smi") or "not found")

COMP = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

print("\n-- competition data --")
for path in sorted(glob.glob(COMP + "/*")):
    print("  %s (%d bytes)" % (path, os.path.getsize(path)))

MODEL_DIR = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"

print("\n-- base model mount --")
print("  expected:", MODEL_DIR)
print("  exists  :", os.path.isdir(MODEL_DIR))
if os.path.isdir(MODEL_DIR):
    entries = sorted(os.listdir(MODEL_DIR))
    print("  entries :", len(entries))
    print("  sample  :", entries[:12])
    total_bytes = sum(
        os.path.getsize(os.path.join(root, name))
        for root, _, names in os.walk(MODEL_DIR)
        for name in names
    )
    print("  size    : %d bytes (%.2f GiB)" % (total_bytes, total_bytes / 1024 ** 3))
else:
    print("  candidates:", sorted(glob.glob("/kaggle/input/models/*/*/*/*"))[:10])

WHEELHOUSE_HINT = "/kaggle/input/arc2-unsloth-wheelhouse"

print("\n-- declared wheelhouse dataset (dataset_sources) --")
print("  expected:", WHEELHOUSE_HINT)
print("  exists  :", os.path.isdir(WHEELHOUSE_HINT))
if os.path.isdir(WHEELHOUSE_HINT):
    entries = sorted(os.listdir(WHEELHOUSE_HINT))
    wheels = [name for name in entries if name.endswith(".whl")]
    total = sum(os.path.getsize(os.path.join(WHEELHOUSE_HINT, name)) for name in entries)
    print("  entries : %d (%d wheels)" % (len(entries), len(wheels)))
    print("  size    : %d bytes (%.1f MiB)" % (total, total / 1024 ** 2))
    pinned = os.path.join(WHEELHOUSE_HINT, "PINNED.txt")
    if os.path.isfile(pinned):
        with open(pinned) as handle:
            print("  PINNED.txt:", len([l for l in handle if l.strip()]), "pins")
else:
    print("  MISSING -> the install cell below will fail loudly")

print("\nOFFLINE PREFLIGHT DONE in %.1fs" % (time.time() - PREFLIGHT_T0))
print("not importable (before):", before_missing if before_missing else "none")
print("=" * 78)


In [ ]:
# ---------------------------------------------------------------------------
# Offline dependency install from the declared wheelhouse dataset.
#
# The Kaggle image does not ship unsloth, unsloth_zoo, trl, bitsandbytes,
# xformers, torchao>=0.13, cut_cross_entropy, tyro, msgspec or structlog at all,
# and it ships transformers 5.0.0, datasets 5.0.0, huggingface_hub 1.11.0,
# dill 0.4.1 and torchao 0.10.0 at versions unsloth's dependency graph rejects.
# `enable_internet: False` forbids PyPI, so the pinned wheels arrive as the
# `dataset_sources` entry `ser8147/arc2-unsloth-wheelhouse` (built by
# `scripts/kaggle/unsloth_wheelhouse.sh`) and are installed with --no-index.
#
# Why the downgrades are unavoidable:
#   transformers    5.0.0 -> 4.57.6    unsloth requires >=4.51.3,<=5.5.0 but
#                                      EXCLUDES 5.0.0 and 5.1.0; trl 0.24.0 also
#                                      needs >=4.56.1, so 4.57.6 is the newest
#                                      4.x that satisfies both
#   datasets        5.0.0 -> 4.3.0     unsloth requires <4.4.0
#   huggingface_hub 1.11.0 -> 0.36.2   transformers 4.57.6 requires <1.0
#   dill            0.4.1 -> 0.4.0     datasets 4.3.0 requires <0.4.1
#
# Everything else the pipeline needs is already in the image at a satisfying
# version and is deliberately NOT pinned: tokenizers 0.22.2, safetensors 0.7.0,
# diffusers 0.37.1, fsspec 2025.3.0, importlib-metadata 8.7.1, typer 0.24.2,
# shellingham 1.5.4, annotated-doc 0.0.4, multiprocess 0.70.16, sentencepiece
# 0.2.1, nest-asyncio 1.6.0, torchvision 0.25.0+cu128, docstring-parser 0.18.0,
# typeguard 4.5.1, hf-xet 1.4.3. Pinning those was the first cut of this cell and
# it was wrong: a pin on fsspec or importlib-metadata needlessly changes a
# package the image already satisfies and trips unrelated pins (gcsfs wants
# exactly fsspec 2025.3.0; opentelemetry-api wants importlib-metadata <8.8.0).
#
# `torch` is deliberately NOT vendored: the image's own 2.10.0+cu128 satisfies
# unsloth's `torch<2.13.0,>=2.4.0`, and republishing it would clobber the
# image's CUDA build.
#
# This cell fails LOUDLY. A missing wheelhouse, a wheelhouse that no longer
# matches PINNED.txt, a non-zero pip exit, a pinned version that did not take
# effect, or a failed import afterwards all raise -- the alternative is a
# silently broken run that still emits a placeholder submission.
# ---------------------------------------------------------------------------
import glob
import importlib
import importlib.metadata as md
import os
import subprocess
import sys
import time

PINNED = [
    # packages the image does not ship at all
    "unsloth==2026.9.11",
    "unsloth_zoo==2026.9.7",
    "trl==0.24.0",
    "bitsandbytes==0.50.2",
    "xformers==0.0.34",
    # packages the image ships, but at a version the pipeline cannot accept
    "transformers==4.57.6",
    "datasets==4.3.0",
    "huggingface_hub==0.36.2",
    "dill==0.4.0",
    "torchao==0.16.0",
    # dependencies of the above that the image does not ship
    "cut_cross_entropy==25.1.1",
    "tyro==1.0.16",
    "msgspec==0.21.1",
    "structlog==26.1.0",
    "hf_transfer==0.1.9",
]

# Locate the mount by its PINNED.txt, so a rename of the dataset surfaces as a
# clear error instead of a mysterious pip failure.
found = sorted(glob.glob("/kaggle/input/*/PINNED.txt"))
matching = [p for p in found if os.path.basename(os.path.dirname(p)) == "arc2-unsloth-wheelhouse"]
if not matching:
    raise RuntimeError(
        "GATE FAILED: the wheelhouse dataset is not mounted. Expected "
        "/kaggle/input/arc2-unsloth-wheelhouse/PINNED.txt; found %s" % found
    )
WHEELHOUSE = os.path.dirname(matching[0])

with open(os.path.join(WHEELHOUSE, "PINNED.txt")) as handle:
    dataset_pins = sorted(line.strip() for line in handle if line.strip())
if dataset_pins != sorted(PINNED):
    only_dataset = sorted(set(dataset_pins) - set(PINNED))
    only_cell = sorted(set(PINNED) - set(dataset_pins))
    raise RuntimeError(
        "GATE FAILED: this cell and the dataset disagree on the pinned set "
        "(only in the dataset: %s; only in this cell: %s). Rebuild with "
        "scripts/kaggle/unsloth_wheelhouse.sh --create." % (only_dataset, only_cell)
    )

wheels = sorted(os.path.basename(p) for p in glob.glob(os.path.join(WHEELHOUSE, "*.whl")))
print("wheelhouse  :", WHEELHOUSE)
print("wheels      :", len(wheels), "for", len(PINNED), "pins")
print("torch       : NOT vendored on purpose; the image's own build is kept")

INSTALL_T0 = time.time()
command = [
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--disable-pip-version-check",
    "--find-links", WHEELHOUSE,
    *PINNED,
]
print("\n$ " + " ".join(command) + "\n")
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
if result.stderr.strip():
    print(result.stderr)
print("*** pip exit code %d after %.0fs" % (result.returncode, time.time() - INSTALL_T0))
if result.returncode != 0:
    raise RuntimeError(
        "GATE FAILED: the offline wheelhouse install returned exit code %d. See the pip output "
        "above; a missing wheel means the dataset and this cell have drifted."
        % result.returncode
    )

importlib.invalidate_caches()

print("\n" + "=" * 78)
print("OFFLINE INSTALL -- AFTER STATE")
print("=" * 78)
after_missing = package_table()

def public(version):
    """PEP 440 public part. `torchvision==0.25.0` and the image's `0.25.0+cu128` are the same
    release, and the local label is the CUDA build we want, so it must not fail the check."""
    return version.split("+", 1)[0]


broken = []
for spec in PINNED:
    name, _, version = spec.partition("==")
    observed = dist_version(name)
    if public(observed) != public(version):
        broken.append("%s pinned to %s but installed %s" % (name, version, observed))
if broken:
    raise RuntimeError("GATE FAILED: pinned versions did not take effect: %s" % broken)
print("\npinned versions in effect: %d/%d" % (len(PINNED), len(PINNED)))

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True
)
print("\n-- pip check (informational; the image also holds packages this pipeline never touches) --")
print((check.stdout + check.stderr).strip() or "(clean)")

# The hard gate: exactly the import arc_solver.py performs, in a child process,
# plus the packages the image was missing. A silent failure here would let a
# broken run continue into the submission cell and emit placeholder grids.
GATE = "\n".join([
    "import transformers, trl, bitsandbytes, peft, datasets, huggingface_hub, tokenizers, safetensors",
    "from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer",
    "import unsloth_zoo, xformers, torch",
    "print('transformers  ', transformers.__version__)",
    "print('trl           ', trl.__version__)",
    "print('bitsandbytes  ', bitsandbytes.__version__)",
    "print('peft          ', peft.__version__)",
    "print('datasets      ', datasets.__version__)",
    "print('huggingface_hub', huggingface_hub.__version__)",
    "print('tokenizers    ', tokenizers.__version__)",
    "print('safetensors   ', safetensors.__version__)",
    "print('unsloth_zoo   ', unsloth_zoo.__version__)",
    "print('xformers      ', xformers.__version__)",
    "print('torch         ', torch.__version__, '| cuda', torch.version.cuda)",
    "print('FastLanguageModel/UnslothTrainingArguments/UnslothTrainer: imported')",
])
print("\n-- hard import gate: the imports arc_solver.py makes --")
gate = subprocess.run([sys.executable, "-c", GATE], capture_output=True, text=True)
print(gate.stdout)
if gate.stderr.strip():
    print(gate.stderr)
if gate.returncode != 0:
    raise RuntimeError(
        "GATE FAILED: the wheelhouse installed but the pipeline's imports do not work "
        "(exit code %d). This pipeline cannot run offline as declared." % gate.returncode
    )

print("\nnot importable (after):", after_missing if after_missing else "none")
print("OFFLINE INSTALL OK in %.0fs" % (time.time() - INSTALL_T0))
print("=" * 78)


In [ ]:
# Reference-kernel offline adaptation of the budget cell.
#
# Upstream (kept byte-for-byte for a competition rerun):
#     global_end_time = time.time() + 12 * 3600 - 600
#
# A commit-mode push is NOT a competition rerun: it is our offline verification.
# There, the budget is bounded so L4x4 consumption stays proportionate. Only the
# number of seconds differs; the worker's end-time semantics are untouched.
import os
import time

IS_RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

# Commit-mode verification budget, in seconds of wall clock from this point.
COMMIT_BUDGET_SECONDS = int(os.getenv("ARC_REFERENCE_COMMIT_BUDGET", "1800"))

if IS_RERUN:
    global_end_time = time.time() + 12 * 3600 - 600
else:
    global_end_time = time.time() + COMMIT_BUDGET_SECONDS

# Run the real competition challenge set (and therefore the real submission
# schema) in both modes. ARC_REFERENCE_TEST_SET=0 restores the upstream
# non-rerun behaviour exactly (4 evaluation tasks + accuracy benchmark).
os.environ.setdefault("ARC_REFERENCE_TEST_SET", "1")

print("rerun mode             :", IS_RERUN)
print("ARC_REFERENCE_TEST_SET :", os.environ["ARC_REFERENCE_TEST_SET"])
print("global_end_time        : %d (+%.0fs)" % (global_end_time, global_end_time - time.time()))


In [ ]:
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def is_horizontal_reflection(g: np.ndarray) -> bool:
    return np.array_equal(g, np.fliplr(g))

def is_vertical_reflection(g: np.ndarray) -> bool:
    return np.array_equal(g, np.flipud(g))

def is_180_rotation(g: np.ndarray) -> bool:
    return np.array_equal(g, np.rot90(g, 2))

def is_main_diagonal_reflection(g: np.ndarray) -> bool:
    if g.shape[0] != g.shape[1]:
        return False
    return np.array_equal(g, g.T)

def is_anti_diagonal_reflection(g: np.ndarray) -> bool:
    if g.shape[0] != g.shape[1]:
        return False
    return np.array_equal(g, np.fliplr(np.flipud(g)).T)

def is_symmetry_trivial(name: str, g: np.ndarray) -> bool:
    h, w = g.shape
    if name == "horizontal":
        return w <= 1
    if name == "vertical":
        return h <= 1
    if name in ("main_diagonal", "anti_diagonal"):
        return h != w or h <= 1
    if name == "rotation_180":
        return h <= 1 and w <= 1
    return False

SYMMETRY_PREDICATES = {
    "horizontal": is_horizontal_reflection,
    "vertical": is_vertical_reflection,
    "rotation_180": is_180_rotation,
    "main_diagonal": is_main_diagonal_reflection,
    "anti_diagonal": is_anti_diagonal_reflection,
}

def get_common_symmetries(train_outputs: list[np.ndarray]) -> set[str]:
    if not train_outputs:
        return set()
    common = set(SYMMETRY_PREDICATES.keys())
    for out in train_outputs:
        common = {name for name in common if SYMMETRY_PREDICATES[name](out)}
        if not common:
            break
    non_trivial = set()
    for name in common:
        if any(not is_symmetry_trivial(name, out) for out in train_outputs):
            non_trivial.add(name)
    return non_trivial

def get_constant_new_colors(train_inputs: list[np.ndarray], train_outputs: list[np.ndarray]) -> set[int]:
    if not train_inputs or not train_outputs:
        return set()
    new_sets = []
    for inp, out in zip(train_inputs, train_outputs):
        in_c = set(np.unique(inp)) - {0}
        out_c = set(np.unique(out)) - {0}
        new_sets.append(out_c - in_c)
    if not new_sets:
        return set()
    common = set(new_sets[0])
    for s in new_sets[1:]:
        common &= s
    return common

def verify_solution_invariants(solution, task_dict):
    """AutoHarness Deterministic Invariant Verifier for ARC candidates (D4 Symmetry + Soft Color Conservation)."""
    if not isinstance(solution, np.ndarray) or solution.ndim != 2:
        return False, -1000.0
    h, w = solution.shape
    if h <= 0 or h > 30 or w <= 0 or w > 30:
        return False, -1000.0

    if task_dict is None:
        return True, 0.0

    train_pairs = task_dict.get("train", [])
    if not train_pairs:
        return True, 0.0

    test_input = np.asarray(task_dict["test"][0]["input"])
    train_inputs = [np.asarray(p["input"]) for p in train_pairs]
    train_outputs = [np.asarray(p["output"]) for p in train_pairs]

    # 1. Palette Invariant: output colors must be subset of input + demonstration palette
    allowed_colors = set(np.unique(test_input))
    for inp, out in zip(train_inputs, train_outputs):
        allowed_colors.update(np.unique(inp))
        allowed_colors.update(np.unique(out))

    sol_colors = set(np.unique(solution))
    if not sol_colors.issubset(allowed_colors):
        return False, -500.0

    # 2. Shape Invariants:
    # A) Exact shape identity
    same_shape = all(out.shape == inp.shape for inp, out in zip(train_inputs, train_outputs))
    if same_shape and solution.shape != test_input.shape:
        return False, -300.0

    # B) Constant output shape across demonstrations
    train_out_shapes = {out.shape for out in train_outputs}
    if len(train_out_shapes) == 1:
        expected_shape = list(train_out_shapes)[0]
        if solution.shape != expected_shape:
            return False, -300.0

    # C) Integer scaling factor
    scale_factors = set()
    scale_valid = True
    for inp, out in zip(train_inputs, train_outputs):
        in_sh = inp.shape
        out_sh = out.shape
        if in_sh[0] > 0 and in_sh[1] > 0 and out_sh[0] % in_sh[0] == 0 and out_sh[1] % in_sh[1] == 0:
            scale_factors.add((out_sh[0] // in_sh[0], out_sh[1] // in_sh[1]))
        else:
            scale_valid = False
            break
    if scale_valid and len(scale_factors) == 1:
        s_r, s_c = list(scale_factors)[0]
        expected_scaled = (test_input.shape[0] * s_r, test_input.shape[1] * s_c)
        if solution.shape != expected_scaled:
            return False, -300.0

    # 3. D4 Dihedral Symmetry Invariants
    common_syms = get_common_symmetries(train_outputs)
    if common_syms:
        for sym_name in common_syms:
            if not SYMMETRY_PREDICATES[sym_name](solution):
                return False, -400.0

    # 4. Soft Color Histogram Conservation
    constant_new = get_constant_new_colors(train_inputs, train_outputs)
    local_allowed = (set(np.unique(test_input)) - {0}) | constant_new | {0}
    bad_cells = sum(1 for cell in solution.flat if cell not in local_allowed)
    color_penalty = (bad_cells / float(h * w)) * 250.0

    # 5. 12-Parameter Quadrature HIHO 0.50 Coherence Reranker
    mask_in = test_input != 0
    mask_sol = solution != 0
    if test_input.shape == solution.shape:
        inter = float(np.sum(mask_in & mask_sol))
        union = float(np.sum(mask_in | mask_sol))
        sigma_space = inter / max(union, 1.0)
    else:
        sigma_space = 0.50

    if test_input.shape == solution.shape:
        sigma_field = float(np.mean(solution == test_input))
    else:
        in_colors = set(np.unique(test_input))
        sol_c = set(np.unique(solution))
        sigma_field = len(in_colors & sol_c) / max(len(in_colors | sol_c), 1)

    sym_count = sum(1 for sym_fn in SYMMETRY_PREDICATES.values() if sym_fn(solution))
    sigma_control = sym_count / float(len(SYMMETRY_PREDICATES))

    sigma_precip = float(np.count_nonzero(solution)) / float(h * w)

    coherence = 0.25 * (sigma_space + sigma_field + sigma_control + sigma_precip)
    phi_hiho = max(0.0, 1.0 - 4.0 * ((coherence - 0.5) ** 2))
    dissonance = abs(coherence - 0.5) * 2.0
    hiho_bonus = 50.0 * phi_hiho - 25.0 * dissonance

    return True, 10.0 + hiho_bonus - color_penalty

def score_sum(guesses, getter, task_dict=None):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)

    ranked = []
    for sc, sol in scores.values():
        base_score = getter(sc)
        if task_dict is not None:
            _, bonus = verify_solution_invariants(sol, task_dict)
            base_score += bonus
        ranked.append((base_score, sol))

    ranked = sorted(ranked, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in ranked]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses, task_dict=None):
    return score_sum(guesses, getter_full_probmul_3, task_dict=task_dict)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses, task_dict=None):
    return score_sum(guesses, getter_kgmon, task_dict=task_dict)

selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]

class ArcDecoder:
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        results = {}
        for bk, v in self.decoded_results.items():
            task_dict = getattr(self.dataset, "queries", {}).get(bk)
            guesses_dict = {k: g for k, g in v.items()}
            results[bk] = selection_algorithm(guesses_dict, task_dict=task_dict)
        return results

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms with AutoHarness Invariants...")
        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0
        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():
            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)
            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():
                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])
                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"
                output_len = f"{solution.shape[0]}x{solution.shape[1]}"
                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ({name})")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=True,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    # Reference-kernel offline adaptation (see starter.py): in commit mode the
    # challenge set is selected by ARC_REFERENCE_TEST_SET so the verification
    # artifact exercises the real competition code path and submission schema.
    use_test_set = bool(rerun_mode) or os.getenv("ARC_REFERENCE_TEST_SET") == "1"

    if use_test_set:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time, num_gpus):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank % num_gpus)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start on CUDA device {rank % num_gpus}!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    # Reference-kernel offline adaptation: the commit-mode artifact must
    # exercise the real competition code path and the real submission schema, so
    # ARC_REFERENCE_TEST_SET=1 uses the test challenge set in both modes. Unset
    # (or 0) restores the upstream non-rerun behaviour exactly.
    use_test_set = bool(rerun_mode) or os.getenv("ARC_REFERENCE_TEST_SET") == "1"

    if use_test_set:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    print("queueing %d task(s) from %s" % (len(data), test_path))
    for key in sorted(data.keys()):
        if not use_test_set:
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        queue.put(key)

    num_gpus = max(1, torch.cuda.device_count()) if torch.cuda.is_available() else 1
    print(f"Detected {num_gpus} CUDA device(s) for starter.py")
    nprocs = min(4, num_gpus)

    for _ in range(nprocs):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time, num_gpus), nprocs=nprocs)


In [ ]:
# Reference-kernel offline adaptation: stream the worker log and make a
# non-zero exit fatal.
#
# Upstream used `!python starter.py --end-time {global_end_time}`. A shell-magic
# failure does not stop the notebook, so a broken run would continue into the
# submission cell and silently emit placeholder grids. Streaming the log through
# a checked subprocess keeps the evidence visible and the failure loud.
import os
import subprocess
import sys
import time

worker_env = dict(os.environ)
worker_env["UNSLOTH_DISABLE_STATISTICS"] = "1"
worker_env["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
worker_env["OMP_NUM_THREADS"] = "12"

WORKER_T0 = time.time()
process = subprocess.Popen(
    [sys.executable, "starter.py", "--end-time", str(global_end_time)],
    env=worker_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
returncode = process.wait()
WORKER_ELAPSED = time.time() - WORKER_T0

print("*** starter.py exit code %d after %.0fs" % (returncode, WORKER_ELAPSED))
if returncode != 0:
    raise RuntimeError(
        "starter.py failed with exit code %d after %.0fs; the pipeline did not run"
        % (returncode, WORKER_ELAPSED)
    )


In [ ]:
import json
import os
import time
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

SUBMISSION_PATH = "/kaggle/working/submission.json"
INFERENCE_OUTPUTS = "/kaggle/inference_outputs"
COMP = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
TEST_CHALLENGES = COMP + "/arc-agi_test_challenges.json"
EVAL_CHALLENGES = COMP + "/arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS = COMP + "/arc-agi_evaluation_solutions.json"

os.makedirs(INFERENCE_OUTPUTS, exist_ok=True)
os.makedirs("/kaggle/working", exist_ok=True)
SUBMIT_T0 = time.time()

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

# Same selection rule as starter.py / arc_solver.worker.
use_test_set = bool(rerun_mode) or os.getenv("ARC_REFERENCE_TEST_SET") == "1"

if use_test_set:
    data = ArcDataset.from_file(TEST_CHALLENGES)
else:
    data = ArcDataset.from_file(EVAL_CHALLENGES)
    data = data.load_replies(EVAL_SOLUTIONS)

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results(INFERENCE_OUTPUTS)

submission = data.get_submission(decoder.run_selection_algo())

with open(SUBMISSION_PATH, "w") as handle:
    json.dump(submission, handle)

print("*** wrote %s (%d bytes)" % (SUBMISSION_PATH, os.path.getsize(SUBMISSION_PATH)))

# Only the evaluation set has public solutions, so the accuracy benchmark can
# only run there. It is skipped whenever we run the competition test set.
if not use_test_set:
    decoder.benchmark_selection_algos()
    with open(SUBMISSION_PATH) as handle:
        print("*** Reload score:", data.validate_submission(json.load(handle)))

# ---------------------------------------------------------------------------
# Submission contract check against arc-agi_test_challenges.json
# ---------------------------------------------------------------------------
with open(TEST_CHALLENGES) as handle:
    test_tasks = json.load(handle)

test_ids = sorted(test_tasks.keys())
problems = []

for key in test_ids:
    if key not in submission:
        problems.append("%s: task id absent from submission" % key)

for key in test_ids:
    outputs = submission.get(key)
    expected = len(test_tasks[key]["test"])
    if not isinstance(outputs, list):
        problems.append("%s: submission value is not a list" % key)
        continue
    if len(outputs) != expected:
        problems.append("%s: %d outputs for %d test inputs" % (key, len(outputs), expected))
    for index, output in enumerate(outputs):
        for attempt in ("attempt_1", "attempt_2"):
            if attempt not in output:
                problems.append("%s[%d]: missing %s" % (key, index, attempt))
                continue
            grid = output[attempt]
            if not isinstance(grid, list) or not grid:
                problems.append("%s[%d].%s: not a non-empty list" % (key, index, attempt))
            elif not all(isinstance(row, list) and row for row in grid):
                problems.append("%s[%d].%s: not a 2-D grid" % (key, index, attempt))
            elif len({len(row) for row in grid}) != 1:
                problems.append("%s[%d].%s: ragged rows" % (key, index, attempt))
            elif not (1 <= len(grid) <= 30 and 1 <= len(grid[0]) <= 30):
                problems.append("%s[%d].%s: out of range %dx%d"
                                % (key, index, attempt, len(grid), len(grid[0])))
            elif any(not isinstance(value, int) or not 0 <= value <= 9
                     for row in grid for value in row):
                problems.append("%s[%d].%s: value outside 0-9" % (key, index, attempt))

total_outputs = 0
non_placeholder = 0
for key in test_ids:
    for output in submission.get(key, []):
        if not isinstance(output, dict):
            continue
        total_outputs += 1
        if output.get("attempt_1") != [[0]] or output.get("attempt_2") != [[0]]:
            non_placeholder += 1

decoded_keys = len(decoder.decoded_results)
decoded_outputs = sum(len(v) for v in decoder.decoded_results.values())

print("\n" + "=" * 78)
print("SUBMISSION CONTRACT CHECK")
print("=" * 78)
print("submission path            :", SUBMISSION_PATH)
print("challenge file             :", TEST_CHALLENGES)
print("test task ids              :", len(test_ids))
print("test outputs (task x input):", total_outputs)
print("attempt keys required      : attempt_1, attempt_2")
print("tasks with decoded results : %d/%d" % (decoded_keys, len(test_ids)))
print("decoded candidate outputs  :", decoded_outputs)
print("outputs with a non-placeholder attempt :", non_placeholder)
print("problems                   :", len(problems))
for problem in problems[:20]:
    print("   -", problem)
if len(problems) > 20:
    print("   ... and %d more" % (len(problems) - 20))
print("VERDICT                    :", "PASS" if not problems else "FAIL")
print("elapsed for submission step: %.1fs" % (time.time() - SUBMIT_T0))
print("=" * 78)

if problems:
    raise RuntimeError("submission.json failed the contract check: %s" % problems[:5])
